# Paper figures

Generates Figures 2-6 and Supplementary Figures S1-S2 from the stored run results in `results/` (written by `scripts/run_scenarios.py` and `scripts/analyze_results.py`) and the hourly inputs in `data/`. Each figure cell exports one vector PDF to `figures/`. Figure 1 is a static system schematic and is not generated here.

Run all cells from the repository root, or use `python scripts/make_figures.py` (optionally `--figure <name>`).

All panels show deterministic model outcomes or hourly inputs, not statistical replicates; no uncertainty bars or significance tests are implied.


In [ ]:
import matplotlib
matplotlib.use("Agg")


In [ ]:
# Shared setup for all figure cells
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch, PathPatch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D

# sci-figure helpers (palette + apply_sci_style + save_pdf), vendored at
# notebooks/sci_figure_helpers.py so the figure pipeline is repo-contained.
_here = Path.cwd().resolve()
_NOTEBOOKS_DIR = _here if _here.name == "notebooks" else _here / "notebooks"
if str(_NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(_NOTEBOOKS_DIR))
from sci_figure_helpers import (  # noqa: E402
    PALETTE,
    apply_sci_style,
    save_pdf,
    add_panel_label,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data"
OUTPUTS = PROJECT_ROOT / "results"
FIGURES = PROJECT_ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

MANUSCRIPT_RC = {
    # Source lettering is intentionally a little larger than the finished
    # 7 pt artwork target because LaTeX scales each PDF to \linewidth.
    "font.size": 8.0,
    "axes.labelsize": 8.0,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 8.0,
}


def apply_manuscript_style(width: str = "ae_single") -> None:
    apply_sci_style(width)
    plt.rcParams.update(MANUSCRIPT_RC)


apply_manuscript_style("ae_single")  # final PDF lettering targets Elsevier 7 pt guidance

BODY_W = 7.0
SINGLE_W = BODY_W
DOUBLE_W = BODY_W
MAIN_FILLS = [
    PALETTE["fill_blue"],
    PALETTE["fill_salmon"],
    PALETTE["fill_orange"],
    PALETTE["fill_gray"],
]
MAIN_STROKES = [
    PALETTE["stroke_navy"],
    PALETTE["stroke_clay"],
    PALETTE["stroke_teal"],
    "#555555",
]
TAC_STACK_VOM_FILL = "#DDF3DE"
TAC_STACK_FUEL_FILL = PALETTE["fill_orange"]
TAC_STACK_C0_IMPORT_FILL = "#C1F4F0"
GRID_EXPORT_FILL = PALETTE["fill_gray"]
GRID_EXPORT_ALPHA = 1.0
BLOCK_EDGE = "#FFFFFF"
BLOCK_EDGE_LW = 0.5
EDGE_LW = BLOCK_EDGE_LW
TEXT_SIZE = MANUSCRIPT_RC["font.size"]
GRID_KW = dict(axis="y", alpha=0.3, linestyle="--", linewidth=0.5)
ZERO_LINE_KW = dict(color="#777777", linewidth=0.75, linestyle="--", alpha=0.9, zorder=10)
LINE_PLOT_LW = 1.85
THIN_LINE_PLOT_LW = 1.45
LINE_MARKER_SIZE = 6.2
LINE_MARKER_AREA = LINE_MARKER_SIZE ** 2
LINE_MARKER_EDGE_WIDTH = 1.0
BAR_LABEL_OFFSET_PT = 3
POINT_LABEL_OFFSET_PT = 5
VALUE_LABEL_OFFSET_PT = BAR_LABEL_OFFSET_PT
FIGURE2_LINE_COLORS = [MAIN_FILLS[1], MAIN_FILLS[0], MAIN_FILLS[3], MAIN_FILLS[2]]
FIGURE6_LINE_COLORS = [MAIN_FILLS[3], MAIN_FILLS[1], MAIN_FILLS[2]]
RENDER_UNUSED_FIGURES = False
BAR_GRADIENT_STEPS = 256


def annotate_vertical_value(ax, x, y, label, *, fontsize=TEXT_SIZE, offset_pt=VALUE_LABEL_OFFSET_PT,
                            color="#000000", force_positive_side=None, zorder=10, x_offset_pt=0):
    """Place a numeric label just outside a vertical bar or point endpoint."""
    sign_positive = y >= 0 if force_positive_side is None else bool(force_positive_side)
    dy = offset_pt if sign_positive else -offset_pt
    va = "bottom" if sign_positive else "top"
    ax.annotate(
        label,
        xy=(x, y), xycoords="data",
        xytext=(x_offset_pt, dy), textcoords="offset points",
        ha="center", va=va,
        fontsize=fontsize, color=color,
        zorder=zorder,
        annotation_clip=False,
    )


def annotate_horizontal_endpoint(ax, x, y, label, *, side, fontsize=TEXT_SIZE,
                                 offset_pt=POINT_LABEL_OFFSET_PT, color="#000000", zorder=10):
    """Place a numeric label just outside a horizontal range endpoint."""
    dx = -offset_pt if side == "left" else offset_pt
    ha = "right" if side == "left" else "left"
    ax.annotate(
        label,
        xy=(x, y), xycoords="data",
        xytext=(dx, 0), textcoords="offset points",
        ha=ha, va="center",
        fontsize=fontsize, color=color,
        zorder=zorder,
        annotation_clip=False,
    )


def maybe_render_unused_figure(render_fn) -> None:
    if RENDER_UNUSED_FIGURES:
        render_fn()


def _bar_gradient_cmap(color: str, steps: int = BAR_GRADIENT_STEPS) -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list("white_to_bar_color", ["#FFFFFF", color], N=steps)


def apply_vertical_bar_gradients(ax, bars, colors, steps: int = BAR_GRADIENT_STEPS,
                                 *, white_at_zero: bool = False, gamma: float = 1.0) -> None:
    if isinstance(colors, str):
        colors = [colors] * len(bars)
    for bar, color in zip(bars, colors):
        height = bar.get_height()
        if abs(height) < 1e-12:
            bar.set_visible(False)
            continue
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        x0 = bar.get_x()
        x1 = x0 + bar.get_width()
        y_start = bar.get_y()
        y_end = y_start + height
        y0, y1 = sorted((y_start, y_end))
        ramp = np.linspace(0.0, 1.0, steps) ** gamma
        if white_at_zero and height < 0:
            ramp = ramp[::-1]
        gradient = np.tile(ramp[:, None], (1, 2))
        ax.imshow(
            gradient,
            extent=(x0, x1, y0, y1),
            origin="lower",
            aspect="auto",
            cmap=_bar_gradient_cmap(color, steps),
            interpolation="bilinear",
            alpha=1.0,
            zorder=bar.get_zorder(),
        )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        bar.set_visible(False)


def draw_horizontal_gradient_bar(ax, y, x_start, x_end, height, color, *, white_at="left",
                                 steps: int = BAR_GRADIENT_STEPS, gamma: float = 1.0) -> None:
    if x_end <= x_start:
        return
    ramp = np.linspace(0.0, 1.0, steps) ** gamma
    if white_at == "right":
        ramp = ramp[::-1]
    gradient = np.tile(ramp[None, :], (2, 1))
    ax.imshow(
        gradient,
        extent=(x_start, x_end, y - height / 2, y + height / 2),
        origin="lower",
        aspect="auto",
        cmap=_bar_gradient_cmap(color, steps),
        interpolation="nearest",
        resample=False,
        zorder=2,
    )


def fill_between_vertical_gradient(ax, x, y_lower, y_upper, color, *, alpha=0.32,
                                   steps: int = BAR_GRADIENT_STEPS, zorder=1.1) -> None:
    x = np.asarray(x, dtype=float)
    y_lower = np.asarray(y_lower, dtype=float)
    y_upper = np.asarray(y_upper, dtype=float)
    ymin = float(np.nanmin(np.minimum(y_lower, y_upper)))
    ymax = float(np.nanmax(np.maximum(y_lower, y_upper)))
    if not np.isfinite(ymin) or not np.isfinite(ymax) or ymax <= ymin:
        return
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    clip = ax.fill_between(x, y_lower, y_upper, facecolor="none", edgecolor="none",
                           linewidth=0, zorder=zorder)
    ramp = np.linspace(0.0, 1.0, steps)
    gradient = np.tile(ramp[:, None], (1, 2))
    im = ax.imshow(
        gradient,
        extent=(float(np.nanmin(x)), float(np.nanmax(x)), ymin, ymax),
        origin="lower",
        aspect="auto",
        cmap=_bar_gradient_cmap(color, steps),
        interpolation="bilinear",
        alpha=alpha,
        zorder=zorder,
    )
    paths = clip.get_paths()
    if paths:
        im.set_clip_path(PathPatch(paths[0], transform=ax.transData))
    clip.remove()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


CASE_LINE = {
    0: MAIN_STROKES[0],
    1: MAIN_STROKES[1],
    2: MAIN_STROKES[2],
    3: MAIN_STROKES[3],
}
CASE_FILL = {
    0: MAIN_FILLS[0],
    1: MAIN_FILLS[1],
    2: MAIN_FILLS[2],
    3: MAIN_FILLS[3],
}
df = pd.read_csv(OUTPUTS / "master_kpi_table.csv")
print(f"Loaded {len(df)} rows from master_kpi_table.csv")
print(df.groupby("group").size())

# Convenience column for absolute Premium percentage
df["premium_pct"] = df["heat_recovery_premium"] * 100

# Case identity mapping for legends and colors
CASE_LABEL = {
    0: "Case 0 — Grid only",
    1: "Case 1 — Nuclear, no recovery",
    2: "Case 2 — Nuclear + cascade extraction",
    3: "Case 3 — NGCC on-site",
}
CASE_COLOR = {
    0: CASE_LINE[0],
    1: CASE_LINE[1],
    2: CASE_LINE[2],
    3: CASE_LINE[3],
}
CASE_MARKER = {0: "o", 1: "s", 2: "D", 3: "^"}

In [ ]:
from matplotlib import transforms
import json
from pathlib import Path

plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['Arial'],
                    'mathtext.fontset':'custom','mathtext.rm':'Arial',
                    'mathtext.it':'Arial:italic','mathtext.bf':'Arial:bold',
                    'svg.fonttype':'none','pdf.fonttype':42})
def display(*args):
    pass

# Output file names follow the figure numbering of the paper.
FIGURE_FILES = {
    'fig_inputs_overview': 'fig2_inputs_overview',
    'fig_baseline_economics': 'fig3_baseline_economics',
    'fig_operation_value': 'fig4_operation_value',
    'fig_sensitivity_boundary_atlas': 'fig5_boundary_atlas',
    'fig_policy_sensitivity_summary': 'fig6_policy_sensitivity',
    'fig12_s8_size_matching': 'figS1_size_matching',
    'fig_sensitivity_cooling_response': 'figS2_cooling_response',
}

def save_pdf(fig,name,outdir):
    name=FIGURE_FILES[name]
    fig.canvas.draw()
    # No creation date, so rebuilt PDFs are byte-identical.
    fig.savefig(FIGURES/(name+'.pdf'),bbox_inches='tight',dpi=400,metadata={'CreationDate': None})
    print('Exported', FIGURES/(name+'.pdf'))


In [ ]:
COMPOSITE_HSPACE = 0.28
FIG2_WSPACE = 0.30
FIG2_HSPACE = 0.48
FIG2_LEFT = 0.12
FIG2_RIGHT = 0.98


# =====================================================================
# Panel builders (each draws onto a given axes)
# =====================================================================

def panel_tac_stack(ax):
    """Draw the TAC stack onto ax and return the legend handles."""
    base = df[df.group == "main_baseline"].sort_values("case_id").reset_index(drop=True)
    cases = base.case_id.astype(int).values
    labels = [f"C{c}" for c in cases]
    capex = base.capex_annual_usd.values / 1e6
    fom = base.fom_annual_usd.values / 1e6
    vom = base.vom_annual_usd.values / 1e6
    fuel = base.fuel_annual_usd.values / 1e6
    grid = base.grid_annual_usd.values / 1e6
    tac = base.tac_usd_per_yr.values / 1e6
    premium = base.premium_pct.values
    grid_pos = np.where(grid > 0, grid, 0.0)
    grid_neg = np.where(grid < 0, grid, 0.0)

    x = np.arange(len(cases))
    bw = 0.62
    bot = np.zeros_like(capex)
    for lab, vals, color, alpha, hatch in [
        ("Capital", capex, MAIN_FILLS[0], 0.92, None),
        ("Fixed O&M", fom, MAIN_FILLS[1], 0.92, None),
        ("Variable O&M", vom, TAC_STACK_VOM_FILL, 0.92, None),
        ("Fuel", fuel, TAC_STACK_FUEL_FILL, 0.88, None),
    ]:
        if vals.sum() < 1e-3:
            continue
        ax.bar(x, vals, bottom=bot, width=bw, color=color, edgecolor=BLOCK_EDGE,
               linewidth=EDGE_LW, alpha=alpha, hatch=hatch, label=lab)
        bot = bot + vals
    c0_import = np.where(cases == 0, grid_pos, 0.0)
    other_import = np.where(cases != 0, grid_pos, 0.0)
    if other_import.sum() >= 1e-3:
        ax.bar(x, other_import, bottom=bot, width=bw, color=MAIN_FILLS[0],
               edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=0.52, hatch="////")
    if c0_import.sum() >= 1e-3:
        ax.bar(x, c0_import, bottom=bot, width=bw, color=TAC_STACK_C0_IMPORT_FILL,
               edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=1.0)
    bot = bot + grid_pos
    if (grid_neg < 0).any():
        ax.bar(x, grid_neg, width=bw, color=GRID_EXPORT_FILL, edgecolor=BLOCK_EDGE,
               linewidth=EDGE_LW, alpha=GRID_EXPORT_ALPHA)
    ptc = np.nan_to_num(base["ptc_annual_usd"].to_numpy(dtype=float)) / 1e6
    if (ptc < 0).any():
        ax.bar(x, ptc, bottom=grid_neg, width=bw, color="#7FB7A6",
               edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=0.85)
    stack_top = capex + fom + vom + fuel + grid_pos
    for xi, tac_i, prem_i, top in zip(x, tac, premium, stack_top):
        ax.scatter(xi, tac_i, marker="_", s=320, color=MAIN_STROKES[3], linewidth=1.6, zorder=5)
        ax.annotate(f"TAC {tac_i:.1f}\nM = {prem_i:+.1f}%", xy=(xi, top),
                    xytext=(0, VALUE_LABEL_OFFSET_PT), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8, color="#000000")
    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(r"Annualized cost (M\$ yr$^{-1}$)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    return [
        Patch(facecolor=MAIN_FILLS[0], edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, label="Capital"),
        Patch(facecolor=MAIN_FILLS[1], edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, label="Fixed O&M"),
        Patch(facecolor=TAC_STACK_VOM_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, label="Variable O&M"),
        Patch(facecolor=TAC_STACK_FUEL_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=0.88, label="Fuel"),
        Patch(facecolor=TAC_STACK_C0_IMPORT_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, label="Import"),
        Patch(facecolor=GRID_EXPORT_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=GRID_EXPORT_ALPHA, label="Export"),
        Patch(facecolor="#7FB7A6", edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=0.85, label="PTC"),
        Line2D([0], [0], color=MAIN_STROKES[3], lw=1.6, label="Net TAC"),
    ]


def _cc_data():
    s2 = df[df.group == "s2_price"].copy().sort_values(["case_id", "year"])
    s2["tac_m"] = s2.tac_usd_per_yr / 1e6
    s2["co2_kt"] = s2.co2_annual_tonnes / 1e3
    return s2


def _finish_local_axes(ax, grid_axis="y"):
    ax.set_axisbelow(True)
    ax.grid(axis=grid_axis, alpha=0.30, linestyle="--", linewidth=0.5)
    ax.tick_params(direction="out", length=3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def panel_cc_scatter(ax):
    """Cost-carbon plane (self-contained legends)."""
    s2 = _cc_data()
    years = [2022, 2023, 2024]
    cases = [0, 1, 2, 3]
    local_case_label = {0: "C0 grid", 1: "C1 SMR", 2: "C2 SMR + cool", 3: "C3 NGCC"}
    local_case_marker = {0: "o", 1: "s", 2: "P", 3: "^"}
    year_color = {2022: PALETTE["fill_blue"], 2023: PALETTE["fill_salmon"], 2024: PALETTE["fill_orange"]}
    for cid in cases:
        sub = s2[s2.case_id == cid].sort_values("year")
        ax.plot(sub.tac_m, sub.co2_kt, color="#B6B6B6", linewidth=THIN_LINE_PLOT_LW, alpha=0.68, zorder=1.5)
        for _, row in sub.iterrows():
            ax.scatter(row.tac_m, row.co2_kt, marker=local_case_marker[cid], s=LINE_MARKER_AREA,
                       facecolor=year_color[int(row.year)], edgecolor=BLOCK_EDGE,
                       linewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)
    ax.set_xlabel(r"TAC (M\$ yr$^{-1}$)")
    ax.set_ylabel(r"Export-credited CO$_2$ balance (kt yr$^{-1}$)")
    ax.set_xlim(20, 260)
    ax.set_ylim(-525, 520)
    _finish_local_axes(ax)
    case_handles = [Line2D([0], [0], marker=local_case_marker[cid], linestyle="none", color="none",
                           markerfacecolor="#777777", markeredgecolor=BLOCK_EDGE,
                           markeredgewidth=LINE_MARKER_EDGE_WIDTH, markersize=LINE_MARKER_SIZE,
                           label=local_case_label[cid]) for cid in cases]
    year_handles = [Line2D([0], [0], marker="o", linestyle="none", color="none",
                           markerfacecolor=year_color[y], markeredgecolor=BLOCK_EDGE,
                           markeredgewidth=LINE_MARKER_EDGE_WIDTH, markersize=LINE_MARKER_SIZE,
                           label=str(y)) for y in years]
    leg_case = ax.legend(handles=case_handles, loc="center", bbox_to_anchor=(0.52, 0.54),
                         ncol=2, frameon=True, facecolor="white", edgecolor="none", framealpha=0.88,
                         fontsize=TEXT_SIZE, handlelength=1.05, handletextpad=0.55, borderpad=0.32,
                         labelspacing=0.36, columnspacing=0.65, borderaxespad=0.0)
    ax.add_artist(leg_case)
    ax.legend(handles=year_handles, loc="upper right", bbox_to_anchor=(0.985, 0.985), ncol=1,
              frameon=True, facecolor="white", edgecolor="none", framealpha=0.88,
              fontsize=TEXT_SIZE, handlelength=0.8, handletextpad=0.35, borderpad=0.28,
              labelspacing=0.30, columnspacing=0.75, borderaxespad=0.0)
    ax.annotate("", xy=(0.055, 0.070), xytext=(0.185, 0.235), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="->", color=PALETTE["stroke_teal"], lw=0.9, alpha=0.72))
    ax.text(0.190, 0.245, "preferred", transform=ax.transAxes, fontsize=TEXT_SIZE,
            color=PALETTE["stroke_teal"], ha="left", va="center")


def panel_cc_accounting(ax):
    """Nuclear CO2 accounting (self-contained legend, top of panel)."""
    s2 = _cc_data()
    years = [2022, 2023, 2024]
    local_case_short = {0: "C0", 1: "C1", 2: "C2", 3: "C3"}
    x_pos, x_labels, grid_import, reactor_lca, export_credit, net_co2 = [], [], [], [], [], []
    xpos = 0.0
    for year in years:
        for cid in (1, 2):
            row = s2[(s2.year == year) & (s2.case_id == cid)].iloc[0]
            x_pos.append(xpos)
            x_labels.append(f"{str(year)[-2:]}\n{local_case_short[cid]}")
            grid_import.append(row.co2_grid_import_tonnes / 1e3)
            reactor_lca.append(row.co2_rx_lifecycle_tonnes / 1e3)
            export_credit.append(-row.co2_export_credit_tonnes / 1e3)
            net_co2.append(row.co2_kt)
            xpos += 1.0
        xpos += 0.45
    x_pos = np.array(x_pos)
    grid_import = np.array(grid_import)
    reactor_lca = np.array(reactor_lca)
    export_credit = np.array(export_credit)
    net_co2 = np.array(net_co2)
    positive_total = grid_import + reactor_lca
    bars_grid = ax.bar(x_pos, grid_import, color=PALETTE["fill_blue"], edgecolor=BLOCK_EDGE,
                       linewidth=BLOCK_EDGE_LW, width=0.70)
    apply_vertical_bar_gradients(ax, bars_grid, PALETTE["fill_blue"])
    bars_lca = ax.bar(x_pos, reactor_lca, bottom=grid_import, color=PALETTE["fill_orange"],
                      edgecolor=BLOCK_EDGE, linewidth=BLOCK_EDGE_LW, width=0.70)
    apply_vertical_bar_gradients(ax, bars_lca, PALETTE["fill_orange"])
    bars_export = ax.bar(x_pos, export_credit, color=PALETTE["fill_salmon"], edgecolor=BLOCK_EDGE,
                         linewidth=BLOCK_EDGE_LW, width=0.70)
    apply_vertical_bar_gradients(ax, bars_export, PALETTE["fill_salmon"], white_at_zero=True, gamma=3.0)
    ax.scatter(x_pos, net_co2, marker="_", s=170, color="#222222", linewidth=1.0, zorder=5)
    for xi, top, net in zip(x_pos, positive_total, net_co2):
        annotate_vertical_value(ax, xi, top, f"+{top:.0f}", color="#111111")
        annotate_vertical_value(
            ax, xi, net, f"{net:.0f}", color="#111111",
            force_positive_side=True,
        )
    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel(r"Export-credited CO$_2$ balance (kt yr$^{-1}$)")
    ax.set_ylim(-570, 185)
    _finish_local_axes(ax)
    handles = [
        Patch(facecolor=PALETTE["fill_blue"], edgecolor=BLOCK_EDGE, label="Grid import"),
        Patch(facecolor=PALETTE["fill_orange"], edgecolor=BLOCK_EDGE, label="Reactor lifecycle"),
        Patch(facecolor=PALETTE["fill_salmon"], edgecolor=BLOCK_EDGE, label="Export credit"),
        Line2D([0], [0], color="#222222", linewidth=1.2, label="Net"),
    ]
    ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.965), ncol=2, frameon=True,
              facecolor="white", edgecolor="none", framealpha=0.88, fontsize=TEXT_SIZE,
              handlelength=0.9, columnspacing=0.8, handletextpad=0.4, borderpad=0.3,
              labelspacing=0.30, borderaxespad=0.0)


def panel_dispatch(ax, week):
    """Dispatch for one week onto ax; returns the 5 legend specs."""
    disp = pd.read_csv(OUTPUTS / "main_baseline" / "case2" / "dispatch.csv.gz")
    starts = {"winter": 24 * 14, "summer": 24 * (31 + 28 + 31 + 30 + 31 + 30 + 14)}
    names = {"winter": "Winter (Jan 15 - Jan 22)", "summer": "Summer (Jul 15 - Jul 22)"}
    start = starts[week]
    sl = disp.iloc[start:start + 168].reset_index(drop=True)
    hours = np.arange(168)
    ax.fill_between(hours, 0, sl.P_turb_net_MW, color=PALETTE["fill_blue"], alpha=0.85,
                    edgecolor="white", linewidth=0.4)
    ax.plot(hours, sl.P_rx_MWth, color=FIGURE6_LINE_COLORS[0], linewidth=THIN_LINE_PLOT_LW)
    ax.set_xlim(0, 167)
    ax.set_xlabel("Hour of week" if week == "summer" else "")
    ax.set_ylabel(r"Power and heat (MW)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax_r = ax.twinx()
    ax_r.plot(hours, sl.Q_to_abs_MWth, color=FIGURE6_LINE_COLORS[1], linewidth=THIN_LINE_PLOT_LW, linestyle="--")
    ax_r.plot(hours, sl.Q_abs_cool_MWth, color=FIGURE6_LINE_COLORS[2], linewidth=THIN_LINE_PLOT_LW)
    ax_r.fill_between(hours, 0, sl.Q_VCC_cool_MWth, color=PALETTE["fill_salmon"], alpha=0.6,
                      edgecolor="white", linewidth=0.3)
    ax_r.set_ylabel(r"Cooling and extraction (MW)")
    ax_r.set_ylim(bottom=0)
    ax_r.spines["right"].set_visible(True)
    ax_r.text(0.40, 1.045, names[week], transform=ax.transAxes, fontsize=8,
              color="#222222", zorder=30, clip_on=False,
              bbox=None)
    return [
        (r"$P_{\mathrm{tn}}$ (MW$_\mathrm{e}$)", "patch", PALETTE["fill_blue"], None),
        (r"$P_{\mathrm{rx}}$ (MW$_\mathrm{th}$)", "line", FIGURE6_LINE_COLORS[0], "-"),
        (r"$Q_{\mathrm{ext}}$ (MW$_\mathrm{th}$)", "line", FIGURE6_LINE_COLORS[1], "--"),
        (r"$Q_{\mathrm{a}}$ (MW$_\mathrm{c}$)", "line", FIGURE6_LINE_COLORS[2], "-"),
        (r"$Q_{\mathrm{v}}$ (MW$_\mathrm{c}$)", "patch", PALETTE["fill_salmon"], None),
    ]


def panel_value_decomp(ax):
    """Value-decomposition waterfall onto ax."""
    vd = pd.read_csv(OUTPUTS / "value_decomp_case2.csv")
    base_row = vd[vd.matching_key == "main_baseline/base"].iloc[0]
    components = [
        ("VCC\nsaved", base_row.vcc_elec_saved_usd_per_yr / 1e6),
        ("Power\nlost", base_row.turbine_gen_lost_usd_per_yr / 1e6),
        ("Absorber\ncapital", base_row.absorption_capex_fom_usd_per_yr / 1e6),
        ("VCC\nbackup", base_row.crystal_cutoff_backup_usd_per_yr / 1e6),
        ("PTC\nforgone", base_row.ptc_forgone_usd_per_yr / 1e6),
    ]
    net = base_row.net_value_of_absorption_usd_per_yr / 1e6
    residual = base_row.residual_usd_per_yr / 1e6
    components.append(("Residual", residual))
    bar_width = 0.58
    cum = 0.0
    x_pos, heights, bottoms, colors = [], [], [], []
    for name, val in components:
        x_pos.append(name)
        heights.append(val)
        bottoms.append(cum)
        colors.append(MAIN_FILLS[3] if name == "Residual" else (MAIN_FILLS[0] if val > 0 else MAIN_FILLS[1]))
        cum += val
    ax.bar(x_pos, heights, bottom=bottoms, color=colors, edgecolor=BLOCK_EDGE,
           linewidth=EDGE_LW, width=bar_width, alpha=0.88)
    x_pos.append("Net vs\nCase 1")
    heights.append(net)
    bottoms.append(0.0)
    net_color = MAIN_FILLS[0] if net > 0 else MAIN_FILLS[1]
    ax.bar(["Net vs\nCase 1"], [net], color=net_color, edgecolor=BLOCK_EDGE,
           linewidth=EDGE_LW, width=bar_width, alpha=0.88)
    for i in range(len(x_pos) - 1):
        level = bottoms[i] + heights[i]
        ax.plot([i + bar_width / 2, i + 1 - bar_width / 2], [level, level], color="#BDBDBD",
                linewidth=0.55, alpha=0.85, solid_capstyle="butt", zorder=2.2)
    small_value_label_offset_pt = VALUE_LABEL_OFFSET_PT
    vfs = 8.0
    for i, (x, h, b) in enumerate(zip(x_pos[:-1], heights[:-1], bottoms[:-1])):
        y = b + h
        if abs(h) >= 0.5:
            if x == "Residual":
                annotate_vertical_value(
                    ax, i, y, f"{h:+.1f}", fontsize=vfs,
                    force_positive_side=False,
                )
            else:
                fps = h >= 0
                if x == "Absorber\ncapital" and y > 0:
                    fps = True
                annotate_vertical_value(ax, i, y, f"{h:+.1f}", fontsize=vfs, force_positive_side=fps)
        else:
            annotate_vertical_value(
                ax, i, y, f"{h:+.2f}", fontsize=vfs,
                offset_pt=small_value_label_offset_pt,
                force_positive_side=h >= 0,
            )
    annotate_vertical_value(ax, len(x_pos) - 1, net, f"{net:+.1f}", fontsize=vfs, color="#000000")
    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_ylabel(r"Δ vs Case 1 TAC (M\$ yr$^{-1}$)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    plt.setp(ax.get_xticklabels(), fontsize=8)
    ax.set_ylim(min(min(bottoms), net, -12.0), max(heights) + 1.2)


def _dispatch_legend(ax_leg, specs):
    handles = []
    for label, kind, color, ls in specs:
        if kind == "patch":
            handles.append(Patch(facecolor=color, edgecolor="white", label=label))
        else:
            handles.append(Line2D([0], [0], color=color, linestyle=ls or "-",
                                  linewidth=THIN_LINE_PLOT_LW, label=label))
    ax_leg.axis("off")
    ax_leg.legend(handles=handles, loc="upper center", ncol=5, frameon=True, facecolor="white",
                  edgecolor="none", framealpha=0.86, fontsize=8, columnspacing=1.2,
                  handlelength=1.5, borderpad=0.22, borderaxespad=0.0)


# =====================================================================
# Figure 4: Seasonal dispatch and absorption value decomposition
# =====================================================================
def build_operation_value(dirs):
    fig = plt.figure(figsize=(BODY_W, 7.05))
    gs = fig.add_gridspec(
        4, 1,
        height_ratios=[1.0, 1.0, 0.055, 1.18],
        left=0.11, right=0.96, top=0.985, bottom=0.080,
        hspace=COMPOSITE_HSPACE,
    )
    ax_w = fig.add_subplot(gs[0])
    ax_s = fig.add_subplot(gs[1])
    ax_leg = fig.add_subplot(gs[2])
    ax_v = fig.add_subplot(gs[3])
    specs = panel_dispatch(ax_w, "winter")
    panel_dispatch(ax_s, "summer")
    _dispatch_legend(ax_leg, specs)
    panel_value_decomp(ax_v)
    add_panel_label(ax_w, "a", x=-0.07, y=1.04)
    add_panel_label(ax_s, "b", x=-0.07, y=1.04)
    add_panel_label(ax_v, "c", x=-0.07, y=1.04)
    for d in dirs:
        save_pdf(fig, "fig_operation_value", str(d))
    plt.close(fig)


# =====================================================================
# Figure 3: Baseline cost structure and carbon balance
# =====================================================================
def build_baseline_economics(dirs):
    fig = plt.figure(figsize=(BODY_W, 5.75))
    gs = fig.add_gridspec(
        2, 2,
        height_ratios=[1.0, 1.15],
        left=FIG2_LEFT, right=FIG2_RIGHT, top=0.965, bottom=0.095,
        hspace=FIG2_HSPACE, wspace=FIG2_WSPACE,
    )
    ax_tac = fig.add_subplot(gs[0, :])
    ax_sc = fig.add_subplot(gs[1, 0])
    ax_ac = fig.add_subplot(gs[1, 1])
    handles = panel_tac_stack(ax_tac)
    ax_tac.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.145),
                  ncol=8, frameon=False, fontsize=8,
                  columnspacing=0.6, handlelength=0.9, handletextpad=0.3, borderaxespad=0.0)
    panel_cc_scatter(ax_sc)
    panel_cc_accounting(ax_ac)
    add_panel_label(ax_tac, "a", x=-0.065, y=1.04)
    add_panel_label(ax_sc, "b", x=-0.14, y=1.045)
    add_panel_label(ax_ac, "c", x=-0.14, y=1.045)
    for d in dirs:
        save_pdf(fig, "fig_baseline_economics", str(d))
    plt.close(fig)



## Figure 2: hourly inputs


In [ ]:
# Figure 2: Hourly input traces driving the optimization (fig_inputs_overview)
# Panels A-D: wet-bulb distribution, ERCOT LMP duration curves, IT load, and absorption COP.
apply_manuscript_style("ae_double")  # full-width for the input-data figure

fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 5.0))


# (a) Houston wet-bulb seasonal box plot ---------------------------------
def figure_2_panel_a(ax):
    wb = pd.read_csv(DATA / "weather" / "houston_ambient_2023.csv")
    wb["time"] = pd.to_datetime(wb["time"])
    wb["month"] = wb["time"].dt.month
    wb["year"] = wb["time"].dt.year
    wb23 = wb[wb["year"] == 2023]

    months = list(range(1, 13))
    data_by_month = [wb23[wb23.month == m].wet_bulb_C.dropna().values for m in months]
    bp = ax.boxplot(
        data_by_month,
        positions=months,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=FIGURE2_LINE_COLORS[2], linewidth=0.85),
        boxprops=dict(linewidth=0.5, edgecolor=BLOCK_EDGE),
        whiskerprops=dict(linewidth=0.5, color=FIGURE2_LINE_COLORS[2]),
        capprops=dict(linewidth=0.5, color=FIGURE2_LINE_COLORS[2]),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(PALETTE["fill_blue"])
        patch.set_alpha(0.7)

    # Mark the LiBr crystallization line (Twb + 5K approach >= 34C => Twb >= 29C)
    ax.axhline(29.0, color=FIGURE2_LINE_COLORS[0], linestyle="--", linewidth=LINE_PLOT_LW,
               label=r"LiBr gate: $T_{wb}\geq29\,^\circ$C")
    # Mark the COP design wet-bulb anchor
    ax.axhline(26.0, color=FIGURE2_LINE_COLORS[1], linestyle=":", linewidth=LINE_PLOT_LW,
               label=r"Design: $T_{wb}=26\,^\circ$C")

    ax.set_xticks(months)
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.set_ylabel(r"Wet-bulb $T_{wb}$ ($^\circ$C)")
    ax.set_xlabel("Month (2023)")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="lower center", bbox_to_anchor=(0.56, 1.02), ncol=2,
              frameon=True, facecolor="white", edgecolor="none", framealpha=0.86,
              fontsize=8, borderpad=0.25, handlelength=1.25,
              columnspacing=0.75,
              borderaxespad=0.0)
    # Figure 2, Panel A: Houston monthly wet-bulb distribution.
    add_panel_label(ax, "a")


# (b) ERCOT LMP duration curve -------------------------------------------
def figure_2_panel_b(ax):
    # Load raw ERCOT Houston Hub duration curves for each modeled year.
    # Do not synthesize missing years: the reported LMP statistics rely
    # on real hourly traces for all LMP-duration values.
    ercot_dir = DATA / "ercot"
    series_by_year = {}
    if ercot_dir.exists():
        for year in (2022, 2023, 2024):
            for candidate in [f"{year}_dam_lmp_houston.csv", f"ercot_dam_{year}.csv", f"dam_{year}.csv", f"{year}.csv"]:
                f = ercot_dir / candidate
                if f.exists():
                    df = pd.read_csv(f)
                    cols = [c for c in df.columns if "houston" in c.lower() or "lmp" in c.lower() or "price" in c.lower()]
                    if cols:
                        series_by_year[year] = df[cols[0]].dropna().values
                        break
    missing_years = [y for y in (2022, 2023, 2024) if y not in series_by_year]
    if missing_years:
        raise FileNotFoundError(f"Missing raw ERCOT LMP traces for years: {missing_years}")

    colors = {2022: FIGURE2_LINE_COLORS[0], 2023: FIGURE2_LINE_COLORS[1], 2024: FIGURE2_LINE_COLORS[2]}
    for year in (2022, 2023, 2024):
        s = np.sort(series_by_year[year])[::-1]
        x = np.arange(len(s)) / len(s) * 100.0
        ax.plot(x, s, color=colors[year], linewidth=LINE_PLOT_LW, label=str(year))

    ax.set_xlabel("Duration of year (%)")
    ax.set_ylabel(r"ERCOT LMP (\$/MWh$_\mathrm{e}$)")
    ax.set_ylim(bottom=-50)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="upper right", frameon=False, fontsize=8, title="ERCOT year")
    # Figure 2, Panel B: ERCOT LMP duration curves.
    add_panel_label(ax, "b")


# (c) IT load weekly profile ---------------------------------------------
def figure_2_panel_c(ax):
    df = pd.read_csv(DATA / "workload" / "dc_200mw_real_60u_2018.csv")
    # Pull a winter week and a summer week
    winter_start = 24 * 14
    summer_start = 24 * (31 + 28 + 31 + 30 + 31 + 30 + 14)  # mid-July
    hours = np.arange(168)
    winter = df.iloc[winter_start:winter_start + 168].IT_load_MW.values
    summer = df.iloc[summer_start:summer_start + 168].IT_load_MW.values
    winter_color = FIGURE2_LINE_COLORS[1]  # cool light blue for winter
    summer_color = FIGURE2_LINE_COLORS[0]  # warm salmon/pink for summer
    ax.plot(hours, winter, color=winter_color, linewidth=THIN_LINE_PLOT_LW,
            label="Winter (Jan 15–22)")
    ax.plot(hours, summer, color=summer_color, linewidth=THIN_LINE_PLOT_LW,
            label="Summer (Jul 15–22)")
    ax.axhline(df.IT_load_MW.mean(), color=FIGURE2_LINE_COLORS[2], linestyle=":", linewidth=LINE_PLOT_LW,
               label=fr"Annual mean {df.IT_load_MW.mean():.1f} MW$_\mathrm{{e}}$")
    ax.axhline(df.IT_load_MW.max(), color=FIGURE2_LINE_COLORS[3], linestyle="--", linewidth=LINE_PLOT_LW,
               label=fr"Annual peak {df.IT_load_MW.max():.1f} MW$_\mathrm{{e}}$")
    ax.set_xlabel("Hour of week")
    ax.set_ylabel(r"IT load $P_{IT}$ (MW$_\mathrm{e}$)")
    ax.set_xlim(0, 167)
    ax.set_ylim(0, 200)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    handles, labels = ax.get_legend_handles_labels()
    order = [0, 2, 1, 3]  # rows: Winter / Summer, then Mean / Peak
    ax.legend([handles[i] for i in order], [labels[i] for i in order],
              loc="upper center", bbox_to_anchor=(0.53, 0.98), ncol=2,
              frameon=True, facecolor="white", edgecolor="none", framealpha=0.86,
              fontsize=8, borderpad=0.25, columnspacing=0.75,
              handlelength=1.3, labelspacing=0.25, borderaxespad=0.0)
    # Figure 2, Panel C: data-center IT-load profile.
    add_panel_label(ax, "c")


# (d) Absorption COP(T_wb) -----------------------------------------------
def figure_2_panel_d(ax):
    Twb = np.linspace(10.0, 30.0, 256)
    # Reproduce builder.py _absorption_cop
    cop_nameplate = 1.30
    cop_houston_baseline = 1.10
    derate = 0.015
    baseline_T_wb = 26.0
    cop_lin = cop_houston_baseline - derate * (Twb - baseline_T_wb)
    cop_effective = np.maximum(np.minimum(cop_lin, cop_nameplate), 0.5)

    # Mask values above crystallization gate (cooling-water inlet >34C; Twb+5K>=34 -> Twb>=29)
    cop_gated = cop_effective.copy()
    cop_gated[Twb >= 29.0] = np.nan

    ax.plot(Twb, cop_effective, color=FIGURE2_LINE_COLORS[0], linewidth=LINE_PLOT_LW,
            linestyle="--", label="COP (without gate)")
    ax.plot(Twb, cop_gated, color=FIGURE2_LINE_COLORS[1], linewidth=LINE_PLOT_LW,
            label="COP (gated, used in model)")

    ax.axvspan(29.0, 30.0, alpha=0.15, color=PALETTE["fill_salmon"], zorder=0)
    ax.text(29.5, 1.18, "VCC\nbackup", ha="center", va="center", fontsize=8,
            color=FIGURE2_LINE_COLORS[0])

    ax.axhline(1.30, color=FIGURE2_LINE_COLORS[2], linestyle=":", linewidth=LINE_PLOT_LW)
    ax.text(11, 1.31, r"Nameplate cap 1.30", fontsize=8, color=FIGURE2_LINE_COLORS[2])
    ax.scatter([26.0], [1.10], s=42, marker="o",
               facecolor=FIGURE2_LINE_COLORS[3], edgecolor=BLOCK_EDGE, linewidth=LINE_MARKER_EDGE_WIDTH,
               zorder=4)
    ax.annotate("design anchor\n(26 $^\\circ$C, COP 1.10)",
                xy=(26.0, 1.10), xytext=(15, 0.88),
                arrowprops=dict(arrowstyle="-", color=FIGURE2_LINE_COLORS[3], linewidth=LINE_PLOT_LW),
                fontsize=8, ha="left", color="#222222")

    ax.set_xlabel(r"Wet-bulb $T_{wb}$ ($^\circ$C)")
    ax.set_ylabel(r"Absorption COP$_a$")
    ax.set_xlim(10, 30)
    ax.set_ylim(0.7, 1.4)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="lower left", frameon=False, fontsize=8)
    # Figure 2, Panel D: absorption-chiller COP curve.
    add_panel_label(ax, "d")


figure_2_panel_a(axes[0, 0])
figure_2_panel_b(axes[0, 1])
figure_2_panel_c(axes[1, 0])
figure_2_panel_d(axes[1, 1])

fig.tight_layout(h_pad=1.7, w_pad=1.2)
save_pdf(fig, "fig_inputs_overview", str(FIGURES))
display(fig)
plt.close(fig)

apply_manuscript_style("ae_single")  # restore the default style for the remaining cells


## Figure 3: baseline cost structure and export-credited carbon balance


In [ ]:
build_baseline_economics([FIGURES])


## Figure 4: seasonal dispatch and absorption cost decomposition


In [ ]:
build_operation_value([FIGURES])


## Figure 5: market, capital and financing boundaries


In [ ]:
# Figure 5: Market, reactor-capital, financing, and joint capital-cost boundaries (fig_sensitivity_boundary_atlas)
# Panels A-D: ERCOT year regime, Reactor capital trajectory, WACC, and joint SMR x absorption CAPEX.
def figure_5_boundary_atlas() -> None:
    fig = plt.figure(figsize=(BODY_W, 6.35))
    gs = fig.add_gridspec(
        3, 2,
        height_ratios=[1.0, 0.88, 1.0],
        left=0.09, right=0.98, top=0.98, bottom=0.09,
        hspace=0.44, wspace=0.30,
    )
    ax_a = fig.add_subplot(gs[0, :])
    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[1, 1])
    ax_d_frame = fig.add_subplot(gs[2, :])
    ax_d_frame.set_axis_off()
    ax_d_frame.patch.set_alpha(0.0)
    ax_d_frame.set_zorder(5)
    d_box = gs[2, :].get_position(fig)
    cbar_gap = 0.012
    cbar_w = 0.018
    cbar_label_space = 0.070
    ax_d = fig.add_axes([
        d_box.x0,
        d_box.y0,
        d_box.width - cbar_gap - cbar_w - cbar_label_space,
        d_box.height,
    ])
    cax_d = fig.add_axes([
        d_box.x1 - cbar_label_space - cbar_w,
        d_box.y0,
        cbar_w,
        d_box.height,
    ])

    case_fill = {
        0: MAIN_FILLS[3],
        1: MAIN_FILLS[0],
        2: MAIN_FILLS[1],
        3: MAIN_FILLS[2],
    }

    def label_bars(ax, bars, vals, fontsize=TEXT_SIZE, outside_below_labels=None, outside_above_labels=None, first_label_x_offset_pt=0):
        for label_index, (b, v) in enumerate(zip(bars, vals)):
            annotate_vertical_value(
                ax,
                b.get_x() + b.get_width() / 2,
                v,
                f"{v:.1f}",
                fontsize=fontsize,
                x_offset_pt=first_label_x_offset_pt if label_index == 0 else 0,
            )

    # a) ERCOT year regime: four cases in each market year.
    s2 = df[df.group == "s2_price"].copy().sort_values(["year", "case_id"])
    years = sorted(s2.year.unique())
    cases = sorted(s2.case_id.unique())
    x = np.arange(len(years))
    bar_w = 0.18
    for i, cid in enumerate(cases):
        vals = [
            s2[(s2.year == y) & (s2.case_id == cid)].premium_pct.iloc[0]
            for y in years
        ]
        offset = (i - (len(cases) - 1) / 2) * bar_w
        bars = ax_a.bar(
            x + offset,
            vals,
            width=bar_w,
            color=case_fill[cid],
            edgecolor=BLOCK_EDGE,
            linewidth=EDGE_LW,
            label=CASE_LABEL[cid].replace(" — ", ": "),
        )
        apply_vertical_bar_gradients(ax_a, bars, case_fill[cid], white_at_zero=True)
        label_bars(ax_a, bars, vals, outside_below_labels={-10, -70}, outside_above_labels={17})
    ax_a.axhline(0, **ZERO_LINE_KW)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels([str(y) for y in years])
    ax_a.set_xlim(x[0] - len(cases) * bar_w / 2 - 0.14,
                  x[-1] + len(cases) * bar_w / 2 + 0.14)
    ax_a.set_xlabel("ERCOT year", labelpad=2)
    ax_a.set_ylabel("Grid-cost margin (%)")
    ax_a.set_ylim(-500, 65)
    ax_a.set_axisbelow(True)
    ax_a.grid(**GRID_KW)
    legend_handles = [
        Patch(facecolor=case_fill[cid], edgecolor=BLOCK_EDGE,
              label=CASE_LABEL[cid].replace(" — ", ": "))
        for cid in cases
    ]
    ax_a.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.01, 0.03),
        ncol=2,
        frameon=False,
        fontsize=8,
        columnspacing=0.8,
        handlelength=1.0,
        borderaxespad=0.0,
    )
    # Figure 5, Panel A: grid-cost margin across ERCOT year regimes.
    add_panel_label(ax_a, "a", x=-0.075, y=1.03)

    # b) Reactor capital trajectory: Case 1 and Case 2 move together, but C2 remains lower.
    s4 = df[df.group == "s4_capex"].copy()
    scen_order = ["FOAK", "ATB_Mid", "NOAK"]
    capex_tick = ["FOAK\n\\$14,700", "ATB-Mid\n\\$7,615", "NOAK\n\\$2,250"]
    xb = np.arange(len(scen_order))
    bar_w2 = 0.34
    for i, cid in enumerate((1, 2)):
        vals = [
            s4[(s4.reactor_scenario == s) & (s4.case_id == cid)].premium_pct.iloc[0]
            for s in scen_order
        ]
        offset = (i - 0.5) * bar_w2
        bars = ax_b.bar(
            xb + offset,
            vals,
            width=bar_w2,
            color=MAIN_FILLS[i],
            edgecolor=BLOCK_EDGE,
            linewidth=EDGE_LW,
            label=f"Case {cid}",
        )
        apply_vertical_bar_gradients(ax_b, bars, MAIN_FILLS[i], white_at_zero=True)
        # Separate the two longer FOAK labels without moving their bar-end anchors.
        label_bars(ax_b, bars, vals, outside_below_labels={-8, -15},
                   first_label_x_offset_pt=-1.5 if i == 0 else 1.5)
    ax_b.axhline(0, **ZERO_LINE_KW)
    ax_b.set_xticks(xb)
    ax_b.set_xticklabels(capex_tick)
    ax_b.set_xlim(xb[0] - bar_w2 - 0.14, xb[-1] + bar_w2 + 0.14)
    ax_b.set_xlabel(r"Reactor capital (\$ kW$_\mathrm{e}^{-1}$)", labelpad=2)
    ax_b.set_ylabel("Grid-cost margin (%)")
    ax_b.set_ylim(-320, 112)
    ax_b.set_axisbelow(True)
    ax_b.grid(**GRID_KW)
    legend_handles = [
        Patch(facecolor=MAIN_FILLS[i], edgecolor=BLOCK_EDGE, label=f"Case {cid}")
        for i, cid in enumerate((1, 2))
    ]
    ax_b.legend(handles=legend_handles, loc="lower right", frameon=False, fontsize=8)
    # Figure 5, Panel B: grid-cost margin across Reactor capital trajectories.
    add_panel_label(ax_b, "b", x=-0.16, y=1.04)

    # c) WACC sweep: a focused financing lever for Case 2.
    s7 = df[df.group == "s7_wacc"].copy().sort_values("wacc_effective")
    wacc_labels = [f"{w * 100:.1f}%" for w in s7.wacc_effective.values]
    vals = s7.premium_pct.values
    base_idx = int(np.argmin(np.abs(s7.wacc_effective.values - 0.067)))
    base_val = vals[base_idx]
    colors = [
        MAIN_FILLS[0] if v > base_val else (MAIN_FILLS[1] if v < base_val else MAIN_FILLS[3])
        for v in vals
    ]
    xc = np.arange(len(vals))
    bars = ax_c.bar(
        xc,
        vals,
        width=0.58,
        color=colors,
        edgecolor=BLOCK_EDGE,
        linewidth=EDGE_LW,
    )
    apply_vertical_bar_gradients(ax_c, bars, colors, white_at_zero=True)
    label_bars(ax_c, bars, vals)
    ax_c.axhline(0, **ZERO_LINE_KW)
    ax_c.set_xticks(xc)
    ax_c.set_xticklabels(wacc_labels)
    ax_c.set_xlim(xc[0] - 0.58 / 2 - 0.12, xc[-1] + 0.58 / 2 + 0.12)
    ax_c.set_xlabel("WACC", labelpad=2)
    ax_c.set_ylabel("Grid-cost margin (%)")
    ax_c.set_ylim(-200, 25)
    ax_c.set_axisbelow(True)
    ax_c.grid(**GRID_KW)
    # Figure 5, Panel C: Case 2 grid-cost margin across WACC.
    add_panel_label(ax_c, "c", x=-0.16, y=1.04)

    # d) Joint SMR and absorption-chiller capital-cost frontier.
    s5 = df[df.group == "s5_feasibility_2d"].copy()
    smr_order = ["NOAK", "Low_Mid", "ATB_Mid", "High_Mid", "FOAK"]
    abs_order = ["Bare_Low", "Mid_Low", "Baseline", "Mid_High", "Turnkey_High"]
    smr_vals = [2250, 5000, 7615, 11000, 14700]
    abs_vals = [450, 600, 750, 900, 1200]
    piv = s5.pivot_table(index="smr_capex_tag", columns="absorption_capex_tag", values="premium_pct")
    piv = piv.reindex(smr_order)[abs_order]
    data = piv.values
    from matplotlib.colors import TwoSlopeNorm
    cmap = LinearSegmentedColormap.from_list(
        "hrp_div", [PALETTE["fill_blue"], "#FFFFFF", PALETTE["fill_salmon"]], N=256
    )
    _vmin = float(np.floor(float(data.min()) / 50.0) * 50.0)
    _vmax = max(30.0, float(np.ceil(float(data.max()) / 10.0) * 10.0))
    norm = TwoSlopeNorm(vmin=_vmin, vcenter=0.0, vmax=_vmax)
    im = ax_d.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax_d.text(j, i, f"{data[i, j]:+.1f}", ha="center", va="center",
                      fontsize=TEXT_SIZE, color="#222222")
    ax_d.set_xticks(np.arange(len(abs_order)))
    ax_d.set_xticklabels([f"{v}" for v in abs_vals])
    ax_d.set_yticks(np.arange(len(smr_order)))
    ax_d.set_yticklabels([f"{t.replace('_', '-')}\n\\${v:,}" for t, v in zip(smr_order, smr_vals)])
    ax_d.set_xlabel(r"Absorber capital (\$ kW$_\mathrm{c}^{-1}$)", labelpad=2)
    ax_d.set_ylabel(r"Reactor capital trajectory (\$ kW$_\mathrm{e}^{-1}$)")
    ax_d.set_xticks(np.arange(-0.5, len(abs_order), 1), minor=True)
    ax_d.set_yticks(np.arange(-0.5, len(smr_order), 1), minor=True)
    ax_d.grid(which="minor", color=BLOCK_EDGE, linestyle="-", linewidth=1.0)
    ax_d.tick_params(which="minor", bottom=False, left=False)
    cb = fig.colorbar(im, cax=cax_d)
    cb.set_label("Case 2 grid-cost margin (%)", labelpad=5)
    cb.set_ticks([_vmin, -200.0, -100.0, 0.0, 30.0, 60.0, _vmax])
    cb.set_ticklabels([f"{int(_vmin):d}", "-200", "-100", "0", "+30", "+60", f"+{int(_vmax):d}"])
    # Figure 5, Panel D: joint SMR and absorption-chiller capital-cost frontier.
    add_panel_label(ax_d_frame, "d", x=0.005, y=1.045)

    # Align Panel D's full visual extent with the panels above. The heatmap's
    # two-line y-tick labels and the colorbar label are wider than the bar
    # panels' numeric ticks, so align by measured tight-bbox overhang rather
    # than by spine: pin D's outer-left to A/B and its outer-right to A/C,
    # which shifts the heatmap inward and lets it compress to fit.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv_fig = fig.transFigure.inverted()
    bb_top = ax_a.get_tightbbox(renderer).transformed(inv_fig)
    bb_heat = ax_d.get_tightbbox(renderer).transformed(inv_fig)
    bb_cbar = cax_d.get_tightbbox(renderer).transformed(inv_fig)
    pos_heat = ax_d.get_position()
    pos_cbar = cax_d.get_position()
    left_pad = pos_heat.x0 - bb_heat.x0     # y-title + tick-label overhang
    right_pad = bb_cbar.x1 - pos_cbar.x1    # colorbar tick-label + label overhang
    new_heat_x0 = bb_top.x0 + left_pad
    new_cbar_x0 = bb_top.x1 - right_pad - cbar_w
    new_heat_x1 = new_cbar_x0 - cbar_gap
    ax_d.set_position([new_heat_x0, pos_heat.y0, new_heat_x1 - new_heat_x0, pos_heat.height])
    cax_d.set_position([new_cbar_x0, pos_cbar.y0, cbar_w, pos_cbar.height])

    save_pdf(fig, "fig_sensitivity_boundary_atlas", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_5_boundary_atlas()


## Figure 6: carbon-price crossover and sensitivity ranking


In [ ]:
# Figure 6: Policy crossover and global sensitivity ranking (fig_policy_sensitivity_summary)
# Panels A-B: carbon-price crossover and ranked one-variable sensitivity.
def figure_6_policy_sensitivity() -> None:
    fig = plt.figure(figsize=(BODY_W, 6.05))
    fig._preserve_project_style = True
    gs = fig.add_gridspec(
        2, 1,
        height_ratios=[1.10, 1.28],
        left=0.15, right=0.98, top=0.98, bottom=0.10,
        hspace=0.22,
    )
    ax_c = fig.add_subplot(gs[0, 0])
    ax_t = fig.add_subplot(gs[1, 0])

    # a) Carbon-price crossover.
    s6 = df[df.group == "s6_carbon_price"].copy().sort_values(["case_id", "carbon_price_usd_per_tco2"])
    panel_a_xmax = 180.0
    metrics = json.loads((OUTPUTS / "paper_metrics.json").read_text())
    crossings: dict[int, float] = {}
    fits: dict[int, tuple[float, float]] = {}
    panel_a_line_colors = FIGURE2_LINE_COLORS.copy()
    panel_a_line_colors[1] = PALETTE["accent_sky"]
    case0_at_x = None
    for cid in (0, 1, 2, 3):
        sub = s6[s6.case_id == cid]
        x = sub.carbon_price_usd_per_tco2.values
        y = sub.tac_usd_per_yr.values / 1e6
        assert np.all(np.diff(x) > 0), "Carbon-price interpolation requires sorted inputs"
        slope = (y[-1] - y[0]) / (x[-1] - x[0])
        intercept = y[0]
        fits[cid] = (intercept, slope)
        line_color = panel_a_line_colors[cid]
        # Keep the solved trajectories solid and the >$100/tCO2
        # extrapolation dashed; whole-panel fading reinforces the distinction.
        x_solid = np.linspace(0.0, 100.0, 101)
        x_dash = np.linspace(100.0, panel_a_xmax, 81)
        ax_c.plot(x_solid, np.interp(x_solid, x, y), color=line_color,
                  linewidth=LINE_PLOT_LW, zorder=2)
        ax_c.plot(x_dash, y[-1] + (y[-1] - y[-2]) / (x[-1] - x[-2]) * (x_dash - x[-1]), color=line_color,
                  linewidth=LINE_PLOT_LW, linestyle=(0, (3.2, 2.2)), zorder=2)
        ax_c.scatter(x, y, marker=CASE_MARKER[cid], s=LINE_MARKER_AREA,
                     facecolor=MAIN_FILLS[3], edgecolor=BLOCK_EDGE,
                     linewidth=LINE_MARKER_EDGE_WIDTH, zorder=4)
        if cid == 0:
            case0_at_x = (intercept, slope)
        else:
            c0_int, c0_slope = case0_at_x
            denom = slope - c0_slope
            if abs(denom) > 1e-9:
                cross_price = (c0_int - intercept) / denom
                if 0 < cross_price < 260:
                    crossings[cid] = cross_price

    # Nuclear crossovers interpolate the reoptimized 50–100 dollar cases.
    crossings[1] = metrics["carbon_crossovers_usd_per_tco2"]["1"]
    crossings[2] = metrics["carbon_crossovers_usd_per_tco2"]["2"]
    c1_cross = crossings[1]
    c2_cross = crossings[2]
    ngcc_cross = crossings[3]

    # The blue condition band has one meaning only: both nuclear cases are
    # less expensive than grid.  It starts at C2 and continues to x = 180.
    ax_c.axvspan(c2_cross, panel_a_xmax, color=PALETTE["fill_blue"],
                 alpha=0.16, lw=0, zorder=0.5)

    # Fade the entire extrapolated side (> $100/tCO2), including trajectories
    # and the blue band, so solved/extrapolated is not confused with a condition.
    ax_c.axvspan(100.0, panel_a_xmax, color="#FFFFFF", alpha=0.58,
                 lw=0, zorder=3)

    threshold_ls = (0, (3.0, 2.2))
    ax_c.axvline(c1_cross, color=PALETTE["stroke_navy"],
                 ls=threshold_ls, lw=1.0, zorder=6)
    ax_c.axvline(c2_cross, color=PALETTE["stroke_teal"],
                 ls=threshold_ls, lw=1.0, zorder=6)
    ax_c.text(c1_cross - 1.8, 0.965, rf"C1 \${c1_cross:.1f}",
              transform=ax_c.get_xaxis_transform(), fontsize=TEXT_SIZE,
              ha="right", va="top", color=PALETTE["stroke_navy"], zorder=7)
    ax_c.text(c2_cross + 1.8, 0.965, rf"C2 \${c2_cross:.1f}",
              transform=ax_c.get_xaxis_transform(), fontsize=TEXT_SIZE,
              ha="left", va="top", color=PALETTE["stroke_teal"], zorder=7)

    ax_c.axvline(ngcc_cross, color="#555555",
                 ls=threshold_ls, lw=0.9, zorder=6)
    ax_c.text(ngcc_cross - 2.0, 0.965,
              rf"NGCC crosses grid" "\n" rf"(\${ngcc_cross:.1f}, extrapolated)",
              transform=ax_c.get_xaxis_transform(), fontsize=TEXT_SIZE,
              ha="right", va="top", color="#555555",
              linespacing=1.05, bbox=dict(facecolor="white", edgecolor="none", pad=0.8), zorder=7)
    ax_c.text(82.0, 0.10, "C1 and C2 < grid",
              transform=ax_c.get_xaxis_transform(), fontsize=TEXT_SIZE,
              ha="center", va="bottom", color=PALETTE["stroke_navy"], zorder=7)

    direct_labels = {
        0: "C0 grid",
        1: "C1 nuclear",
        2: "C2 absorption",
        3: "C3 NGCC",
    }
    label_offsets = {0: 9.0, 1: -20.0, 2: 7.0, 3: -5.0}
    direct_label_colors = panel_a_line_colors.copy()
    direct_label_colors[2] = "#6F6F6F"
    for cid, text in direct_labels.items():
        intercept, slope = fits[cid]
        label_x = 5.0
        label_y = intercept + slope * label_x + label_offsets[cid]
        ax_c.text(label_x, label_y, text,
                  ha="left", va="center", fontsize=TEXT_SIZE,
                  color=direct_label_colors[cid], zorder=7)
    ax_c.set_xlabel(r"Carbon price (\$ tCO$_2^{-1}$)")
    ax_c.set_ylabel("TAC (M\$ yr$^{-1}$)")
    ax_c.set_xlim(0, panel_a_xmax)
    # Corrected C1 extrapolation reaches about42 M\\$ yr$^{-1}$ at180 dollars/t.
    ax_c.set_ylim(35, 155)
    ax_c.set_axisbelow(True)
    ax_c.grid(**GRID_KW)
    # Figure 6, Panel A: TAC extrapolated against carbon price.
    add_panel_label(ax_c, "a", x=-0.12, y=1.05)

    # b) Ranked tornado sensitivity.
    def hrp_percent(run_id: str) -> float:
        return float(df.loc[df.run_id == run_id, "heat_recovery_premium"].iloc[0] * 100.0)

    baseline = hrp_percent("case2_ATB_Mid")
    axes_def = [
        ("Reactor capital", "case2_FOAK", "case2_NOAK"),
        ("Market year", "case2_year2024", "case2_year2022"),
        ("WACC", "case2_wacc_100", "case2_wacc_50"),
        ("Carbon price", "case2_co2_0", "case2_co2_100"),
        ("Data-center size", "case2_load_x050", "case2_load_x300"),
        ("Absorber capital", "case2_smr_ATB_Mid_abs_Turnkey_High", "case2_smr_ATB_Mid_abs_Bare_Low"),
        ("PUE", "case2_pue150", "case2_pue110"),
        ("BESS on/off", "case2_bess_off", "case2_bess_on"),
    ]
    axis_spans = []
    for label, run_a, run_b in axes_def:
        va = hrp_percent(run_a)
        vb = hrp_percent(run_b)
        lo, hi = min(va, vb), max(va, vb)
        axis_spans.append((label, lo, hi, hi - lo))
    axis_spans.sort(key=lambda row: row[3], reverse=True)

    labels = [row[0] for row in axis_spans]
    lows = [row[1] for row in axis_spans]
    highs = [row[2] for row in axis_spans]
    spans = [row[3] for row in axis_spans]
    ypos = np.arange(len(axis_spans))[::-1]
    segment_lengths = []
    for lo, hi in zip(lows, highs):
        left = min(lo, baseline)
        right = max(hi, baseline)
        if left < baseline:
            segment_lengths.append(baseline - left)
        if right > baseline:
            segment_lengths.append(right - baseline)
    max_segment_length = max(segment_lengths) if segment_lengths else 0.0

    def gamma_from_length(length: float, max_gamma: float = 3.5) -> float:
        if max_segment_length <= 0:
            return 1.0
        return 1.0 + (max_gamma - 1.0) * (length / max_segment_length)
    worse_color = MAIN_FILLS[0]
    better_color = MAIN_FILLS[1]
    for y, lo, hi in zip(ypos, lows, highs):
        left = min(lo, baseline)
        right = max(hi, baseline)
        if left < baseline:
            draw_horizontal_gradient_bar(
                ax_t, y, left, baseline, 0.62, worse_color,
                white_at="right", gamma=gamma_from_length(baseline - left),
            )
        if right > baseline:
            draw_horizontal_gradient_bar(
                ax_t, y, baseline, right, 0.62, better_color,
                white_at="left", gamma=gamma_from_length(right - baseline),
            )
        annotate_horizontal_endpoint(
            ax_t, lo, y, f"{lo:.1f}", side="left", color=MAIN_STROKES[0]
        )
        annotate_horizontal_endpoint(
            ax_t, hi, y, f"{hi:.1f}", side="right", color=MAIN_STROKES[1]
        )
    ax_t.axvline(baseline, color="#333333", lw=1.0, ls="--", zorder=8)
    ax_t.annotate(
        f"ATB-Mid baseline ({baseline:.1f})",
        xy=(baseline, 1.0), xycoords=ax_t.get_xaxis_transform(),
        xytext=(POINT_LABEL_OFFSET_PT, VALUE_LABEL_OFFSET_PT), textcoords="offset points",
        ha="left", va="bottom", fontsize=TEXT_SIZE, color="#333333",
        annotation_clip=False, zorder=9,
    )
    ax_t.set_yticks(ypos)
    ax_t.set_yticklabels([f"{label}\n(span {span:.1f} pp)" for label, span in zip(labels, spans)])
    ax_t.set_xlabel("Case 2 grid-cost margin (%)")
    ax_t.set_xlim(min(lows) - 90, max(highs) + 40)
    ax_t.set_ylim(-1.0, len(axis_spans) - 0.4)
    ax_t.set_axisbelow(True)
    ax_t.grid(axis="x", alpha=0.25, linestyle="--", linewidth=0.5)
    ax_t.tick_params(direction="out", length=3)
    for spine in ("top", "right"):
        ax_t.spines[spine].set_visible(False)
    handles = [
        Patch(facecolor=worse_color, edgecolor="white", label="more negative than baseline"),
        Patch(facecolor=better_color, edgecolor="white", label="less negative than baseline"),
    ]
    ax_t.legend(handles=handles, loc="lower left", frameon=True, facecolor="white",
                edgecolor="none", framealpha=1.0, fontsize=8,
                bbox_to_anchor=(0.0, 0.02))
    # Figure 6, Panel B: ranked sensitivity of Case 2 grid-cost margin.
    add_panel_label(ax_t, "b", x=-0.12, y=1.03)

    payload = {
        "source_table": "results/master_kpi_table.csv",
        "panel_a": {
            "case_colors": {str(i): panel_a_line_colors[i] for i in range(4)},
            "solved_range_usd_per_tonne": [0, 100],
            "displayed_range_usd_per_tonne": [0, panel_a_xmax],
            "crossovers_usd_per_tonne": {str(k): float(v) for k, v in crossings.items()},
            "solved_points": s6[["case_id", "carbon_price_usd_per_tco2", "tac_usd_per_yr"]].to_dict("records"),
            "interpolation": "piecewise linear through all three reoptimized points",
            "extrapolation": "last solved segment, 50 to100 dollars per tonne",
        },
        "panel_b": {
            "baseline_margin_pct": baseline,
            "worse_color": worse_color, "better_color": better_color,
            "axes": [{"axis": label, "low_margin_pct": lo, "high_margin_pct": hi, "span_pp": span}
                     for label, lo, hi, span in axis_spans],
        },
    }
    (FIGURES / "fig6_policy_sensitivity_data.json").write_text(json.dumps(payload, indent=2))
    save_pdf(fig, "fig_policy_sensitivity_summary", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_6_policy_sensitivity()


## Supplementary Figure S1: data-center size matching


In [ ]:
# Supplementary Figure S1: Data-center size matching sensitivity (fig12_s8_size_matching)
# Panels A-D: premium, TAC gap, IT-cost intensity, and net grid export.
def figure_s1_size_matching() -> None:
    s8 = df[df.group == "s8_size_matching"].copy()
    if s8.empty:
        print("[fig12] missing s8_size_matching rows; skipping")
        return

    s8["avg_it_mw"] = s8.it_energy_annual_MWh / 8760.0
    s8["net_export_twh"] = (
        s8.P_grid_sell_annual_MWh.fillna(0.0)
        - s8.P_grid_buy_annual_MWh.fillna(0.0)
    ) / 1e6
    s8["tac_myr"] = s8.tac_usd_per_yr / 1e6
    s8["grid_myr"] = s8.tac_case0_baseline_usd_per_yr / 1e6
    s8["gap_myr"] = s8.tac_myr - s8.grid_myr
    s8["tac_per_mwh_it"] = s8.tac_usd_per_yr / s8.it_energy_annual_MWh
    s8["grid_per_mwh_it"] = s8.tac_case0_baseline_usd_per_yr / s8.it_energy_annual_MWh
    s8["gap_per_mwh_it"] = s8.tac_per_mwh_it - s8.grid_per_mwh_it

    fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 4.85), sharex=True)

    line_palette = {
        0: PALETTE["fill_blue"],
        1: PALETTE["fill_blue"],
        2: PALETTE["fill_salmon"],
        3: PALETTE["fill_orange"],
    }
    marker_face = "#7A7A7A"
    marker_edge = "#FFFFFF"
    line_lw = LINE_PLOT_LW
    marker_size = 5.6
    marker_edge_lw = 1.05
    fig4_marker = {1: "X", 2: "D", 3: "^"}
    x_tick_data = (
        s8[s8.case_id == 2]
        .sort_values("load_multiplier")[["load_multiplier", "avg_it_mw"]]
        .drop_duplicates()
    )
    x_ticks = x_tick_data.avg_it_mw.to_numpy()
    x_tick_labels = [f"{x:.1f}" for x in x_ticks]

    def plot_line(ax, x, y, cid, label, marker=None, **kwargs):
        ax.plot(
            x,
            y,
            color=line_palette[cid],
            linewidth=line_lw,
            marker=marker or CASE_MARKER.get(cid, "o"),
            markersize=marker_size,
            markerfacecolor=marker_face,
            markeredgecolor=marker_edge,
            markeredgewidth=marker_edge_lw,
            label=label,
            **kwargs,
        )

    # (a) Cost premium as the campus grows against a fixed BWRX-300.
    ax = axes[0, 0]
    for cid in (1, 2, 3):
        sub = s8[s8.case_id == cid].sort_values("load_multiplier")
        style = {}
        if cid == 1:
            style.update(linestyle="--", zorder=5)
        elif cid == 2:
            style.update(zorder=4)
        else:
            style.update(zorder=3)
        plot_line(
            ax,
            sub.avg_it_mw,
            sub.premium_pct,
            cid,
            CASE_LABEL[cid].replace(" — ", "\n"),
            marker=fig4_marker[cid],
            **style,
        )
    ax.axhline(0, **ZERO_LINE_KW)
    ax.axvline(94.7, color=PALETTE["accent_sky"], linewidth=0.9,
               linestyle="--", alpha=0.95)
    ax.text(106.0, -360, "1.0x load", ha="center", va="center",
            fontsize=8, color="#333333", rotation=90,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=0.6))
    ax.set_ylabel("Grid-cost margin (%)")
    ax.set_xlim(35, 300)
    ax.set_ylim(-465, 45)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="lower right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure S1, Panel A: grid-cost margin versus campus scale.
    add_panel_label(ax, "a")

    # (b) Absolute TAC rises with size, but so does the matching grid baseline.
    ax = axes[0, 1]
    c2 = s8[s8.case_id == 2].sort_values("load_multiplier")
    plot_line(ax, c2.avg_it_mw, c2.grid_myr, 1, "Grid-only baseline", marker="o")
    plot_line(ax, c2.avg_it_mw, c2.tac_myr, 2, "Case 2 TAC", marker="D")
    fill_between_vertical_gradient(ax, c2.avg_it_mw, c2.grid_myr, c2.tac_myr,
                                   PALETTE["fill_salmon"], alpha=0.32)
    gap_25_myr = c2.loc[c2.load_multiplier == 2.5, "gap_myr"].iloc[0]
    gap_row_25 = c2.loc[c2.load_multiplier == 2.5].iloc[0]
    gap_x = gap_row_25["avg_it_mw"]
    gap_grid = gap_row_25["grid_myr"]
    gap_tac = gap_row_25["tac_myr"]
    gap_arrow_x = gap_x + 12.0
    gap_arrow_grid = float(np.interp(gap_arrow_x, c2.avg_it_mw, c2.grid_myr))
    gap_arrow_tac = float(np.interp(gap_arrow_x, c2.avg_it_mw, c2.tac_myr))
    ax.annotate(
        "",
        xy=(gap_arrow_x, gap_arrow_grid),
        xytext=(gap_arrow_x, gap_arrow_tac),
        textcoords="data",
        arrowprops=dict(arrowstyle="<->", color="#555555", lw=0.75,
                        shrinkA=0, shrinkB=0),
        zorder=28,
        annotation_clip=False,
    )
    ax.text(
        gap_x - 58.0,
        gap_tac + 22.0,
        f"Gap at 2.5x\n{gap_25_myr:.1f} M\$ yr$^{{-1}}$",
        ha="left",
        va="center",
        fontsize=TEXT_SIZE,
        color="#333333",
        zorder=30,
        clip_on=False,
    )
    ax.set_ylabel("Annual TAC (M\\$ yr$^{-1}$)")
    ax.set_ylim(0, max(c2.tac_myr.max(), c2.grid_myr.max()) * 1.08)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper left", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure S1, Panel B: Case 2 TAC versus matching grid-only TAC.
    add_panel_label(ax, "b")

    # (c) Unit-load cost intensity reveals the utilization-efficiency gain.
    ax = axes[1, 0]
    plot_line(ax, c2.avg_it_mw, c2.grid_per_mwh_it, 1, "Grid-only baseline", marker="o")
    plot_line(ax, c2.avg_it_mw, c2.tac_per_mwh_it, 2, "Case 2 TAC", marker="D")
    fill_between_vertical_gradient(ax, c2.avg_it_mw, c2.grid_per_mwh_it, c2.tac_per_mwh_it,
                                   PALETTE["fill_salmon"], alpha=0.32)
    gap_1_intensity = c2.loc[c2.load_multiplier == 1.0, "gap_per_mwh_it"].iloc[0]
    gap_30_intensity = c2.loc[c2.load_multiplier == 3.0, "gap_per_mwh_it"].iloc[0]
    ax.annotate(
        f"{gap_1_intensity:.1f} -> {gap_30_intensity:.1f}\ncost gap",
        xy=(c2.loc[c2.load_multiplier == 3.0, "avg_it_mw"].iloc[0],
            c2.loc[c2.load_multiplier == 3.0, "tac_per_mwh_it"].iloc[0]),
        xytext=(-12, 36),
        textcoords="offset points",
        fontsize=TEXT_SIZE,
        color="#333333",
        ha="right",
        va="center",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7),
    )
    ax.set_xlabel(r"Mean IT load (MW$_\mathrm{e}$)")
    ax.set_ylabel(r"Cost intensity (\$ MWh$_\mathrm{IT}^{-1}$)")
    ax.set_ylim(70, 220)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure S1, Panel C: cost intensity per MWh of IT load.
    add_panel_label(ax, "c")

    # (d) Merchant export falls as the campus absorbs more of the fixed unit.
    ax = axes[1, 1]
    for cid in (1, 2):
        sub = s8[s8.case_id == cid].sort_values("load_multiplier")
        plot_line(ax, sub.avg_it_mw, sub.net_export_twh, cid, f"C{cid}", marker=fig4_marker[cid])
    ax.axhline(0, **ZERO_LINE_KW)
    balance = s8[(s8.case_id == 2) & (s8.load_multiplier == 2.0)].iloc[0]
    ax.axvline(balance.avg_it_mw, color=PALETTE["accent_sky"], linewidth=0.9,
               linestyle="--", alpha=0.95)
    ax.annotate(
        "2.0x load",
        xy=(balance.avg_it_mw, balance.net_export_twh),
        xytext=(248, 0.86),
        textcoords="data",
        fontsize=TEXT_SIZE,
        color="#333333",
        ha="left",
        va="center",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7),
    )
    ax.set_xlabel(r"Mean IT load (MW$_\mathrm{e}$)")
    ax.set_ylabel("Net grid export (TWh yr$^{-1}$)")
    ax.set_ylim(-1.10, 2.08)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure S1, Panel D: net grid export versus campus scale.
    add_panel_label(ax, "d")

    for ax in axes.flat:
        ax.set_xlim(35, 300)
        ax.set_xticks(x_ticks)
        ax.tick_params(direction="out", length=3)
    for ax in axes[1, :]:
        ax.set_xticklabels(x_tick_labels)

    fig.tight_layout(w_pad=0.95, h_pad=0.90)
    save_pdf(fig, "fig12_s8_size_matching", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_s1_size_matching()


## Supplementary Figure S2: cooling-side response


In [ ]:
# Supplementary Figure S2: Cooling-side response of the absorption configuration (fig_sensitivity_cooling_response)
# Panels A-D: C2-C1 TAC gap, cooling-electricity saving, PUE premium gap, and carbon-price premium gap.
def figure_s2_cooling_response() -> None:
    pue_levels = [1.10, 1.30, 1.50]
    s1 = df[df.group == "s1_pue"].copy()
    s6 = df[df.group == "s6_carbon_price"].copy()

    tac_gap = []
    pue_gap = []
    cooling_elec_saving = []
    vd = pd.read_csv(OUTPUTS / "value_decomp_case2.csv")
    for pue in pue_levels:
        c1 = s1[(s1.case_id == 1) & (s1.pue.round(2) == pue)].iloc[0]
        c2 = s1[(s1.case_id == 2) & (s1.pue.round(2) == pue)].iloc[0]
        vrow = vd[vd.matching_key == f"s1_pue/_pue{int(round(pue * 100)):03d}"].iloc[0]
        tac_gap.append((c2.tac_usd_per_yr - c1.tac_usd_per_yr) / 1e6)
        pue_gap.append((c2.heat_recovery_premium - c1.heat_recovery_premium) * 100.0)
        # Explicit net column prevents accidentally subtracting Case 2 backup/top-up twice.
        cooling_elec_saving.append(vrow.net_vcc_elec_saved_usd_per_yr / 1e6)

    co2_levels = [0.0, 50.0, 100.0]
    co2_gap = []
    for price in co2_levels:
        c1 = s6[(s6.case_id == 1) & (s6.carbon_price_usd_per_tco2 == price)].iloc[0]
        c2 = s6[(s6.case_id == 2) & (s6.carbon_price_usd_per_tco2 == price)].iloc[0]
        co2_gap.append((c2.heat_recovery_premium - c1.heat_recovery_premium) * 100.0)

    fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 4.65), constrained_layout=True)
    ax_a, ax_b, ax_c, ax_d = axes.ravel()
    x = np.arange(len(pue_levels))
    xtick = [f"{p:.2f}" for p in pue_levels]

    tac_colors = [MAIN_FILLS[1] if v >= 0 else MAIN_FILLS[0] for v in tac_gap]
    bars = ax_a.bar(x, tac_gap, color=tac_colors, edgecolor=BLOCK_EDGE,
                    linewidth=EDGE_LW, width=0.58)
    apply_vertical_bar_gradients(ax_a, bars, tac_colors, white_at_zero=True)
    for b, v in zip(bars, tac_gap):
        annotate_vertical_value(ax_a, b.get_x() + b.get_width() / 2, v, f"{v:.1f}", fontsize=8)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(xtick)
    ax_a.set_xlim(-0.5, len(pue_levels) - 0.5)
    ax_a.set_xlabel("PUE")
    ax_a.set_ylabel("C2 - C1 TAC (M\\$ yr$^{-1}$)")
    ax_a.set_ylim(min(tac_gap) - 2.8, max(tac_gap) + 2.2)
    # Figure S2, Panel A: Case 2 minus Case 1 TAC penalty across PUE.
    add_panel_label(ax_a, "a", x=-0.13, y=1.04)

    bars = ax_b.bar(x, cooling_elec_saving, color=MAIN_FILLS[0], edgecolor=BLOCK_EDGE,
                    linewidth=EDGE_LW, width=0.58)
    apply_vertical_bar_gradients(ax_b, bars, MAIN_FILLS[0])
    for b, v in zip(bars, cooling_elec_saving):
        label_v = 0.0 if abs(v) < 0.05 else v
        annotate_vertical_value(
            ax_b, b.get_x() + b.get_width() / 2, label_v, f"{label_v:.1f}",
            fontsize=8, force_positive_side=label_v >= 0,
        )
    ax_b.set_xticks(x)
    ax_b.set_xticklabels(xtick)
    ax_b.set_xlim(-0.5, len(pue_levels) - 0.5)
    ax_b.set_xlabel("PUE")
    ax_b.set_ylabel("Net cooling-electricity\nsaving (M\\$ yr$^{-1}$)")
    ax_b.set_ylim(-1.5, max(cooling_elec_saving) * 1.28)
    # Figure S2, Panel B: VCC electricity saving net of backup top-up in Case 2.
    add_panel_label(ax_b, "b", x=-0.13, y=1.04)

    line_blue = PALETTE["accent_sky"]
    line_pink = PALETTE["fill_salmon"]
    marker_gray = PALETTE["fill_gray"]
    ax_c.plot(x, pue_gap, "-", color=line_blue, linewidth=LINE_PLOT_LW,
              marker="o", markersize=LINE_MARKER_SIZE, markerfacecolor=marker_gray,
              markeredgecolor=BLOCK_EDGE, markeredgewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)
    ax_c.axhline(0, **ZERO_LINE_KW)
    for xi, yi in zip(x, pue_gap):
        annotate_vertical_value(ax_c, xi, yi, f"{yi:+.1f}", fontsize=8, offset_pt=POINT_LABEL_OFFSET_PT, force_positive_side=(xi == 2))
    ax_c.set_xticks(x)
    ax_c.set_xticklabels(xtick)
    ax_c.set_xlabel("PUE")
    ax_c.set_ylabel("C2 - C1 margin gap (pp)")
    ax_c.set_xlim(-0.35, len(pue_levels) - 0.65)
    # Figure S2, Panel C: Case 2 minus Case 1 premium gap across PUE.
    add_panel_label(ax_c, "c", x=-0.13, y=1.04)

    x2 = np.arange(len(co2_levels))
    ax_d.plot(x2, co2_gap, "-", color=line_pink, linewidth=LINE_PLOT_LW,
              marker="o", markersize=LINE_MARKER_SIZE, markerfacecolor=marker_gray,
              markeredgecolor=BLOCK_EDGE, markeredgewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)
    ax_d.axhline(0, **ZERO_LINE_KW)
    for xi, yi in zip(x2, co2_gap):
        annotate_vertical_value(ax_d, xi, yi, f"{yi:+.1f}", fontsize=8, offset_pt=POINT_LABEL_OFFSET_PT, force_positive_side=(xi == 2))
    ax_d.set_xticks(x2)
    ax_d.set_xticklabels([f"{int(c)}" for c in co2_levels])
    ax_d.set_xlabel(r"Carbon price (\$ tCO$_2^{-1}$)")
    ax_d.set_ylabel("C2 - C1 margin gap (pp)")
    ax_d.set_xlim(-0.35, len(co2_levels) - 0.65)
    # Figure S2, Panel D: Case 2 minus Case 1 premium gap across carbon price.
    add_panel_label(ax_d, "d", x=-0.13, y=1.04)

    ymin = min(min(pue_gap), min(co2_gap)) - 3.8
    ymax = max(max(pue_gap), max(co2_gap)) + 2.4
    for ax in (ax_c, ax_d):
        ax.set_ylim(ymin, ymax)

    for ax in axes.ravel():
        ax.set_axisbelow(True)
        ax.grid(**GRID_KW)
        ax.tick_params(direction="out", length=3)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    save_pdf(fig, "fig_sensitivity_cooling_response", str(FIGURES))
    display(fig)
    plt.close(fig)


figure_s2_cooling_response()